# EIA Plant-Level Coal Capacity Factor, 2001-2024

This notebook builds a plant-year coal capacity-factor panel for 2001-2024. Capacity comes from EIA-860 operable coal generators, while generation comes from EIA-906/920 for 2001-2007 and EIA-923 for 2008-2024 Page 1 plant-fuel generation. DCE is then calculated at the plant-year level.


In [ ]:
from __future__ import annotations

import calendar
import fnmatch
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT_NAME = "Data_center_and_fossil_energy_Replication"
current = Path.cwd().resolve()
while current.name != PROJECT_ROOT_NAME and current.parent != current:
    current = current.parent
if current.name != PROJECT_ROOT_NAME:
    raise RuntimeError(
        f"Could not find {PROJECT_ROOT_NAME}. Run this notebook from inside the project repository."
    )

PROJECT_ROOT = current
RAW = PROJECT_ROOT / "Data" / "raw"
TEMP = PROJECT_ROOT / "Data" / "temp"

EIA_ROOT = RAW / "eia"
F860_ROOT = EIA_ROOT / "f860"
F923_ROOT = EIA_ROOT / "f923"
OUTPUT_DIR = TEMP
STATE_CONTROLS = TEMP / "eia_state_year_controls_2001_2024.csv"
AI_INPUT = RAW / "SPGlobal_Export.xlsx"
GADM_INPUT = RAW / "gadm_410.gpkg"
ELECZONE_INPUT = RAW / "world.geojson"

REBUILD_EIA_BASE_PANEL = True

OUTPUT_CSV = OUTPUT_DIR / "eia_coal_plant_year_generation_mechanism_2001_2024_dce_controls.csv"
OUTPUT_DTA = OUTPUT_DIR / "eia_coal_plant_year_generation_mechanism_2001_2024_dce_controls.dta"
BASE_PANEL_CSV = OUTPUT_DIR / "eia_coal_plant_year_generation_mechanism_2001_2024_base.csv"
COVERAGE_CSV = OUTPUT_DIR / "eia_coal_plant_year_generation_mechanism_2001_2024_coverage.csv"
DCE_COVERAGE_CSV = OUTPUT_DIR / "eia_coal_plant_year_generation_mechanism_2001_2024_dce_coverage.csv"

ANALYSIS_YEARS = list(range(2001, 2025))
COAL_CODES = {"ANT", "BIT", "LIG", "RC", "SC", "SGC", "SUB", "WC"}
OPERATING_STATUS_CODES = {"OP", "OA", "OS"}
LARGER_DC_TYPES = {
    "Hyperscale Data Center",
    "Cloud Data Center",
    "Wholesale Data Center",
    "Crypto Mining Data Center",
}
TIME_WINDOWS = {
    "before06": (None, 2005),
    "06_15": (2006, 2015),
    "16_19": (2016, 2019),
    "20_24": (2020, 2024),
}
BUFFER_DISTANCES_KM = [15, 25, 50, 100, 200]
MIN_DISTANCE_KM = 0.1

print(f"Project root: {PROJECT_ROOT}")
print(f"EIA root: {EIA_ROOT}")
print(f"Output: {OUTPUT_DTA}")


In [8]:
def clean_col(x) -> str:
    return re.sub(r"\s+", " ", str(x).replace("\n", " ")).strip()


def key_col(x) -> str:
    return re.sub(r"[^a-z0-9]+", "", clean_col(x).lower())


def norm_plant(x) -> str:
    if pd.isna(x):
        return ""
    try:
        return str(int(float(x)))
    except Exception:
        return str(x).strip()


def hours_in_year(year: int) -> int:
    return 8784 if calendar.isleap(int(year)) else 8760


def first_col(df: pd.DataFrame, alternatives: list[str], required: bool = True) -> str | None:
    lookup = {key_col(c): c for c in df.columns}
    for alt in alternatives:
        k = key_col(alt)
        if k in lookup:
            return lookup[k]
    for alt in alternatives:
        k = key_col(alt)
        for kk, col in lookup.items():
            if k and (k in kk or kk in k):
                return col
    if required:
        raise KeyError(
            "Missing required column. Tried: "
            + ", ".join(alternatives)
            + f". Available columns include: {list(df.columns)[:40]}"
        )
    return None


def read_excel_header_scan(path: Path, sheet_name=0, required: list[str] | None = None, max_header: int = 25) -> pd.DataFrame:
    required = required or []
    for header in range(max_header + 1):
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, header=header)
        except Exception:
            continue
        df.columns = [clean_col(c) for c in df.columns]
        keys = " ".join(key_col(c) for c in df.columns)
        if all(key_col(r) in keys for r in required):
            return df
    raise ValueError(f"Could not find header in {path} sheet={sheet_name}")


def read_dbf(path: Path) -> pd.DataFrame:
    try:
        from dbfread import DBF
    except ImportError as exc:
        raise ImportError("Reading early EIA-860 DBF files requires dbfread. Install with: pip install dbfread") from exc
    return pd.DataFrame(iter(DBF(str(path), load=True, char_decode_errors="ignore")))


def find_file(folder: Path, patterns: list[str]) -> Path:
    files = [
        p for p in folder.iterdir()
        if p.is_file() and not p.name.startswith(("._", "~$"))
    ]
    for pattern in patterns:
        pattern_lower = pattern.lower()
        matches = [
            p for p in files
            if fnmatch.fnmatch(p.name.lower(), pattern_lower)
        ]
        if matches:
            return sorted(matches)[0]
    raise FileNotFoundError(f"No file in {folder} matching {patterns}")


def eia860_folder(year: int) -> Path:
    folder = F860_ROOT / f"eia860{year}"
    if not folder.exists():
        raise FileNotFoundError(folder)
    return folder


def eia923_source(year: int) -> Path:
    if year == 2001:
        return F923_ROOT / "f906920y2001.xls"
    if year == 2002:
        return F923_ROOT / "f906920y2002.xls"
    if year == 2003:
        return F923_ROOT / "f906920_2003.xls"
    if 2004 <= year <= 2007:
        return F923_ROOT / f"f906920_{year}" / f"f906920_{year}.xls"
    folder = F923_ROOT / f"f923_{year}"
    if year == 2008:
        return find_file(folder, ["eia923December2008.xls"])
    if year in [2009, 2010]:
        return find_file(folder, ["EIA923 SCHEDULES 2_3_4_5*.xls"])
    return find_file(folder, ["EIA923_Schedules_2_3_4_5*.xlsx"])


def distance_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


In [9]:
def read_eia860_generators(year: int) -> pd.DataFrame:
    folder = eia860_folder(year)
    source = "EIA-860 generator"

    if 2001 <= year <= 2003:
        yy = str(year)[-2:]
        path = find_file(folder, [f"GENY{yy}.dbf", f"GENY{yy}.DBF"])
        df = read_dbf(path)
    elif 2004 <= year <= 2008:
        yy = str(year)[-2:]
        path = find_file(folder, [f"GenY{yy}.xls", f"GENY{yy}.xls", f"GeneratorY{yy}.xls"])
        # These files have the variable names in the first row (for example,
        # PLNTCODE, GENCODE, NAMEPLATE and ENERGY_SOURCE_1). Header scanning can
        # mistake title rows for headers, so read them directly.
        df = pd.read_excel(path, sheet_name=0, header=0)
    elif year == 2009:
        path = find_file(folder, ["GeneratorY09.xls"])
        df = pd.read_excel(path, sheet_name="Exist", header=0)
    elif year == 2010:
        path = find_file(folder, ["GeneratorsY2010.xls"])
        df = read_excel_header_scan(path, required=["plant", "gen"], max_header=20)
    elif year == 2011:
        path = find_file(folder, ["GeneratorY2011.xlsx"])
        df = read_excel_header_scan(path, required=["plant", "gen"], max_header=20)
    elif year == 2012:
        path = find_file(folder, ["GeneratorY2012.xlsx"])
        df = read_excel_header_scan(path, required=["plant", "gen"], max_header=20)
    else:
        path = find_file(folder, [f"3_1_Generator_Y{year}.xlsx"])
        df = pd.read_excel(path, sheet_name="Operable", header=1)

    df.columns = [clean_col(c) for c in df.columns]
    plant_col = first_col(df, ["Plant Code", "Plant ID", "Plant", "PLNTCODE", "PLANT"])
    cap_col = first_col(df, ["Nameplate Capacity (MW)", "Nameplate Capacity", "NAMEPLATE", "NAMEPLATE_CAPACITY_MW"])
    fuel_col = first_col(df, ["Energy Source 1", "ENERGY SOURCE 1", "ENSOURCE1", "Prime Mover Energy Source 1"], required=False)
    status_col = first_col(df, ["Status", "Operating Status", "STATUS"], required=False)

    out = pd.DataFrame({
        "plant_code": df[plant_col].map(norm_plant),
        "nameplate_mw": pd.to_numeric(df[cap_col], errors="coerce"),
        "energy_source_1": df[fuel_col].astype(str).str.strip().str.upper() if fuel_col else "",
        "generator_status": df[status_col].astype(str).str.strip().str.upper() if status_col else "",
        "year": int(year),
        "capacity_source": source,
    })
    out = out[out["plant_code"].ne("")]
    out = out[out["energy_source_1"].isin(COAL_CODES)]

    # From 2013 onward we read the Operable sheet directly. For earlier files,
    # keep explicit operating codes when available; if status is absent, retain
    # the row and document coverage in the annual table.
    if out["generator_status"].replace("", np.nan).notna().any():
        out = out[out["generator_status"].isin(OPERATING_STATUS_CODES)]

    out = out[out["nameplate_mw"].notna() & (out["nameplate_mw"] > 0)]
    plant = (
        out.groupby(["plant_code", "year"], as_index=False)
        .agg(
            coal_nameplate_mw=("nameplate_mw", "sum"),
            coal_generator_count=("nameplate_mw", "size"),
            capacity_source=("capacity_source", "first"),
        )
    )
    return plant


def read_eia860_plants(year: int) -> pd.DataFrame:
    folder = eia860_folder(year)
    if 2001 <= year <= 2003:
        yy = str(year)[-2:]
        path = find_file(folder, [f"PLANTY{yy}.DBF", f"PLANTY{yy}.dbf"])
        df = read_dbf(path)
    elif 2004 <= year <= 2008:
        yy = str(year)[-2:]
        path = find_file(folder, [f"PlantY{yy}.xls", f"PLANTY{yy}.xls"])
        df = read_excel_header_scan(path, required=["plant"], max_header=20)
    elif year == 2009:
        path = find_file(folder, ["PlantY09.xls"])
        df = read_excel_header_scan(path, required=["plant"], max_header=20)
    elif year == 2010:
        path = find_file(folder, ["PlantY2010.xls"])
        df = read_excel_header_scan(path, required=["plant"], max_header=20)
    elif year == 2011:
        path = find_file(folder, ["PlantY2011.xlsx", "Plant.xlsx"])
        df = read_excel_header_scan(path, required=["plant"], max_header=20)
    elif year == 2012:
        path = find_file(folder, ["PlantY2012.xlsx"])
        df = read_excel_header_scan(path, required=["plant"], max_header=20)
    else:
        path = find_file(folder, [f"2___Plant_Y{year}.xlsx"])
        df = pd.read_excel(path, sheet_name="Plant", header=1)

    df.columns = [clean_col(c) for c in df.columns]
    plant_col = first_col(df, ["Plant Code", "Plant ID", "Plant", "PLNTCODE", "PLANT"])
    name_col = first_col(df, ["Plant Name", "PLANT NAME", "PLNTNAME"], required=False)
    state_col = first_col(df, ["State", "Plant State", "PSTATABB", "STATE"], required=False)
    lat_col = first_col(df, ["Latitude", "LATITUDE", "LAT", "Plant Latitude"], required=False)
    lon_col = first_col(df, ["Longitude", "LONGITUDE", "LON", "Plant Longitude"], required=False)
    ba_col = first_col(df, ["Balancing Authority Code", "BA Code", "BACODE"], required=False)

    out = pd.DataFrame({
        "plant_code": df[plant_col].map(norm_plant),
        "plant_name": df[name_col].astype(str).str.strip() if name_col else "",
        "state": df[state_col].astype(str).str.strip().str.upper() if state_col else "",
        "latitude": pd.to_numeric(df[lat_col], errors="coerce") if lat_col else np.nan,
        "longitude": pd.to_numeric(df[lon_col], errors="coerce") if lon_col else np.nan,
        "balancing_authority": df[ba_col].astype(str).str.strip() if ba_col else "",
    })
    out = out[out["plant_code"].ne("")]
    return out.drop_duplicates("plant_code")


In [10]:
def read_generation_fuel_page(year: int) -> pd.DataFrame:
    path = eia923_source(year)
    xl = pd.ExcelFile(path)
    sheet = None
    for s in xl.sheet_names:
        ks = key_col(s)
        if "page1" in ks and ("generation" in ks or "fuel" in ks):
            sheet = s
            break
    if sheet is None:
        sheet = xl.sheet_names[0]

    df = read_excel_header_scan(path, sheet_name=sheet, required=["Plant ID", "Reported Fuel Type Code"], max_header=25)
    df.columns = [clean_col(c) for c in df.columns]

    plant_col = first_col(df, ["Plant Id", "Plant ID", "Plant Code", "Plant", "PLNTCODE"])
    fuel_col = first_col(df, ["Reported Fuel Type Code", "Fuel Type Code", "Energy Source", "FUEL", "FUELTYPE", "ENERGY_SOURCE"], required=False)
    state_col = first_col(df, ["Plant State", "State"], required=False)

    month_tokens = [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec",
        "january", "february", "march", "april", "june",
        "july", "august", "september", "october", "november", "december",
    ]

    def is_monthly_net_col(col) -> bool:
        k = key_col(col)
        if "yeartodate" in k or "ytd" in k:
            return False
        is_net = k.startswith("netgen") or "netgeneration" in k
        has_month = any(m in k for m in month_tokens)
        return is_net and has_month

    # Early EIA-906/920 files report monthly net generation columns
    # (NETGEN_JAN ... NETGEN_DEC). Modern EIA-923 files often report an
    # annual net-generation column. Prefer monthly sums when available.
    net_cols = [c for c in df.columns if is_monthly_net_col(c)]
    if net_cols:
        for c in net_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        netgen = df[net_cols].sum(axis=1, min_count=1)
    else:
        annual_candidates = [
            "Net Generation (Megawatthours)",
            "Net Generation (MWh)",
            "Net Generation",
            "YEAR-TO-DATE Net Generation",
            "NETGEN",
            "Netgen",
        ]
        annual_col = first_col(df, annual_candidates, required=False)
        if annual_col is not None:
            netgen = pd.to_numeric(df[annual_col], errors="coerce")
        else:
            raise KeyError(f"Could not identify net generation columns in {path} sheet={sheet}")

    out = pd.DataFrame({
        "plant_code": df[plant_col].map(norm_plant),
        "fuel_code": df[fuel_col].astype(str).str.strip().str.upper() if fuel_col else "",
        "state_from_generation": df[state_col].astype(str).str.strip().str.upper() if state_col else "",
        "coal_net_generation_mwh": netgen,
        "year": int(year),
        "generation_source": "EIA-906/920" if year <= 2007 else "EIA-923",
    })
    out = out[out["plant_code"].ne("")]
    out = out[out["plant_code"].ne("99999")]
    if fuel_col:
        out = out[out["fuel_code"].isin(COAL_CODES)]
    out = out[out["coal_net_generation_mwh"].notna()]
    out = (
        out.groupby(["plant_code", "year"], as_index=False)
        .agg(
            coal_net_generation_mwh=("coal_net_generation_mwh", "sum"),
            generation_source=("generation_source", "first"),
            state_from_generation=("state_from_generation", "first"),
        )
    )
    return out


def build_plant_year(year: int) -> pd.DataFrame:
    cap = read_eia860_generators(year)
    plant = read_eia860_plants(year)
    gen = read_generation_fuel_page(year)
    df = cap.merge(plant, on="plant_code", how="left")
    df = df.merge(gen, on=["plant_code", "year"], how="left")
    df["state"] = df["state"].replace("", np.nan).fillna(df["state_from_generation"])
    df["hours_in_year"] = hours_in_year(year)
    df["plant_capacity_factor"] = df["coal_net_generation_mwh"] / (df["coal_nameplate_mw"] * df["hours_in_year"])
    df["plant_cf_clipped_0_1"] = df["plant_capacity_factor"].clip(lower=0, upper=1)
    df["plant_cf_regression_sample"] = (
        df["plant_capacity_factor"].notna()
        & df["coal_nameplate_mw"].gt(0)
        & df["state"].notna()
    ).astype(int)
    return df


def build_base_panel() -> pd.DataFrame:
    yearly = []
    coverage = []
    for year in ANALYSIS_YEARS:
        print(f"Building plant-year capacity factor for {year}")
        try:
            df = build_plant_year(year)
            yearly.append(df)
            coverage.append({
                "year": year,
                "status": "ok",
                "rows": len(df),
                "plants_with_generation": int(df["coal_net_generation_mwh"].notna().sum()),
                "regression_rows": int(df["plant_cf_regression_sample"].sum()),
                "capacity_source": df["capacity_source"].dropna().iloc[0] if df["capacity_source"].notna().any() else "",
                "generation_source": df["generation_source"].dropna().iloc[0] if df["generation_source"].notna().any() else "",
            })
        except Exception as exc:
            coverage.append({"year": year, "status": f"error: {exc}", "rows": 0})
            print(f"  ERROR {year}: {exc}")
    if not yearly:
        raise RuntimeError("No years were successfully built.")
    panel = pd.concat(yearly, ignore_index=True, sort=False)
    pd.DataFrame(coverage).to_csv(COVERAGE_CSV, index=False)
    panel.to_csv(BASE_PANEL_CSV, index=False)
    return panel


In [11]:
def load_data_centers() -> pd.DataFrame:
    dc = pd.read_excel(AI_INPUT)
    dc.columns = [clean_col(c) for c in dc.columns]
    lat_col = first_col(dc, ["Latitude", "LATITUDE", "Lat"])
    lon_col = first_col(dc, ["Longitude", "LONGITUDE", "Lon"])
    year_col = first_col(dc, ["YR_BUILT", "Year Built", "Commissioning year", "Commission Year", "Built Year"])
    type_col = first_col(dc, ["SECONDARY_PPTY_TYPE", "Property Type", "PROPERTY_TYPE", "Data Center Type", "Type"])
    out = pd.DataFrame({
        "dc_id": np.arange(len(dc), dtype=int),
        "dc_lat": pd.to_numeric(dc[lat_col], errors="coerce"),
        "dc_lon": pd.to_numeric(dc[lon_col], errors="coerce"),
        "dc_year": pd.to_numeric(dc[year_col], errors="coerce"),
        "dc_type": dc[type_col].astype(str).str.strip(),
    })
    out = out.dropna(subset=["dc_lat", "dc_lon", "dc_year"])
    out["dc_year"] = out["dc_year"].astype(int)
    out = out[out["dc_year"] <= 2024].copy()
    out["dc_group"] = np.where(out["dc_type"].isin(LARGER_DC_TYPES), "larger", "other")
    return out.reset_index(drop=True)


def exposure_value(years, weights, groups, plant_year, window, group):
    lo, hi = TIME_WINDOWS[window]
    mask = years <= plant_year
    if lo is not None:
        mask &= years >= lo
    if hi is not None:
        mask &= years <= hi
    if group != "all":
        mask &= groups == group
    if not np.any(mask):
        return 0.0, 0
    return float(np.sum(weights[mask])), 1



def assign_stable_plant_locations(panel: pd.DataFrame) -> pd.DataFrame:
    coords = panel[["plant_code", "year", "latitude", "longitude"]].copy()
    coords = coords.dropna(subset=["latitude", "longitude"])
    coords = coords[coords["plant_code"].ne("")]

    if coords.empty:
        panel["stable_latitude"] = np.nan
        panel["stable_longitude"] = np.nan
        panel["stable_location_count"] = np.nan
        return panel

    stable = (
        coords.groupby(["plant_code", "latitude", "longitude"], as_index=False)
        .agg(
            stable_location_count=("year", "size"),
            stable_location_latest_year=("year", "max"),
        )
        .sort_values(
            ["plant_code", "stable_location_count", "stable_location_latest_year"],
            ascending=[True, False, False],
        )
        .drop_duplicates("plant_code", keep="first")
        .rename(columns={
            "latitude": "stable_latitude",
            "longitude": "stable_longitude",
        })
    )

    return panel.merge(
        stable[[
            "plant_code",
            "stable_latitude",
            "stable_longitude",
            "stable_location_count",
            "stable_location_latest_year",
        ]],
        on="plant_code",
        how="left",
        validate="many_to_one",
    )


def calculate_buffer_dce(plant_year: pd.DataFrame, dc: pd.DataFrame) -> pd.DataFrame:
    dc_lat = dc["dc_lat"].to_numpy()
    dc_lon = dc["dc_lon"].to_numpy()
    dc_year = dc["dc_year"].to_numpy()
    dc_group = dc["dc_group"].to_numpy()
    records = []
    for row in plant_year.itertuples(index=False):
        rec = {"plant_code": row.plant_code, "year": int(row.year)}
        if not np.isfinite(row.latitude) or not np.isfinite(row.longitude):
            for b in BUFFER_DISTANCES_KM:
                for group in ["all", "larger", "other"]:
                    for window in TIME_WINDOWS:
                        rec[f"dce_{group}_{b}km_{window}"] = np.nan
                        rec[f"exp_{group}_{b}km_{window}"] = np.nan
            records.append(rec)
            continue
        dist = distance_km(row.latitude, row.longitude, dc_lat, dc_lon)
        for b in BUFFER_DISTANCES_KM:
            within = dist <= float(b)
            if within.any():
                weights = 1.0 / (np.maximum(dist[within], MIN_DISTANCE_KM) / 10.0)
                years = dc_year[within]
                groups = dc_group[within]
            else:
                weights = np.array([])
                years = np.array([])
                groups = np.array([])
            for window in TIME_WINDOWS:
                for group in ["all", "larger", "other"]:
                    val, exposed = exposure_value(years, weights, groups, row.year, window, group)
                    rec[f"dce_{group}_{b}km_{window}"] = val
                    rec[f"exp_{group}_{b}km_{window}"] = exposed
        records.append(rec)
    return pd.DataFrame(records)




def spatial_join_polygon_keys(points: pd.DataFrame, lat_col: str, lon_col: str, id_col: str,
                              polygon_gdf: gpd.GeoDataFrame, key_cols: list[str], label: str) -> pd.DataFrame:
    point_df = (
        points[[id_col, lat_col, lon_col]]
        .dropna(subset=[lat_col, lon_col])
        .drop_duplicates(subset=[id_col])
        .copy()
    )
    if point_df.empty:
        out = points[[id_col]].drop_duplicates().copy()
        for col in key_cols:
            out[col] = np.nan
        return out

    point_gdf = gpd.GeoDataFrame(
        point_df,
        geometry=gpd.points_from_xy(point_df[lon_col], point_df[lat_col]),
        crs="EPSG:4326",
    )
    if polygon_gdf.crs != point_gdf.crs:
        polygon_gdf = polygon_gdf.to_crs(point_gdf.crs)

    keep_cols = [c for c in key_cols if c in polygon_gdf.columns] + ["geometry"]
    joined = gpd.sjoin(point_gdf, polygon_gdf[keep_cols], how="left", predicate="within")
    matched_ids = set(joined.loc[joined.get(key_cols[0]).notna(), id_col]) if key_cols else set()
    unmatched = point_gdf.loc[~point_gdf[id_col].isin(matched_ids)].copy()
    if len(unmatched) > 0:
        boundary_join = gpd.sjoin(unmatched, polygon_gdf[keep_cols], how="left", predicate="intersects")
        joined = pd.concat(
            [joined.loc[~joined[id_col].isin(unmatched[id_col])], boundary_join],
            ignore_index=True,
        )

    if "_zone_area_km2" in joined.columns:
        joined["_sort_area"] = joined["_zone_area_km2"].fillna(np.inf)
        sort_cols = [id_col, "_sort_area"]
        ascending = [True, True]
    else:
        sort_cols = [id_col]
        ascending = [True]
    joined = joined.sort_values(sort_cols, ascending=ascending).drop_duplicates(id_col, keep="first")
    matched = int(joined[key_cols[0]].notna().sum()) if key_cols and key_cols[0] in joined.columns else 0
    print(f"  {label}: matched {matched:,}/{len(point_gdf):,}")
    return pd.DataFrame(joined[[id_col] + [c for c in key_cols if c in joined.columns]])


def add_spatial_region_keys(plant_year: pd.DataFrame, dc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    plant_points = plant_year[["plant_code", "latitude", "longitude"]].drop_duplicates("plant_code").copy()
    dc_points = dc[["dc_id", "dc_lat", "dc_lon"]].drop_duplicates("dc_id").copy()

    print("Loading GADM boundaries for GID-1/GID-2 exposure")
    gadm = gpd.read_file(GADM_INPUT)
    plant_gadm = spatial_join_polygon_keys(
        plant_points, "latitude", "longitude", "plant_code", gadm,
        ["GID_1", "GID_2"], "EIA plants to GADM",
    ).rename(columns={"GID_1": "plant_gid1", "GID_2": "plant_gid2"})
    dc_gadm = spatial_join_polygon_keys(
        dc_points, "dc_lat", "dc_lon", "dc_id", gadm,
        ["GID_1", "GID_2"], "data centers to GADM",
    ).rename(columns={"GID_1": "dc_gid1", "GID_2": "dc_gid2"})

    print("Loading Electricity Maps zone boundaries for electricity-zone exposure")
    zones = gpd.read_file(ELECZONE_INPUT)
    invalid_n = int((~zones.geometry.is_valid).sum())
    if invalid_n > 0:
        zones["geometry"] = zones.geometry.make_valid()
    zones = zones.loc[zones.geometry.notna() & ~zones.geometry.is_empty].copy()
    zones_equal_area = zones.to_crs("EPSG:6933")
    zones["_zone_area_km2"] = zones_equal_area.geometry.area.to_numpy() / 1_000_000

    plant_zone = spatial_join_polygon_keys(
        plant_points, "latitude", "longitude", "plant_code", zones,
        ["zoneName", "countryKey", "countryName", "_zone_area_km2"],
        "EIA plants to electricity zones",
    ).rename(columns={"zoneName": "plant_ezone"})
    dc_zone = spatial_join_polygon_keys(
        dc_points, "dc_lat", "dc_lon", "dc_id", zones,
        ["zoneName", "countryKey", "countryName", "_zone_area_km2"],
        "data centers to electricity zones",
    ).rename(columns={"zoneName": "dc_ezone"})

    plant_year = plant_year.merge(plant_gadm, on="plant_code", how="left", validate="many_to_one")
    plant_year = plant_year.merge(plant_zone[["plant_code", "plant_ezone"]], on="plant_code", how="left", validate="many_to_one")
    dc = dc.merge(dc_gadm, on="dc_id", how="left", validate="one_to_one")
    dc = dc.merge(dc_zone[["dc_id", "dc_ezone"]], on="dc_id", how="left", validate="one_to_one")
    return plant_year, dc


def calculate_region_dce(plant_year: pd.DataFrame, dc: pd.DataFrame) -> pd.DataFrame:
    region_specs = [
        ("gid1", "plant_gid1", "dc_gid1"),
        ("gid2", "plant_gid2", "dc_gid2"),
        ("ezone", "plant_ezone", "dc_ezone"),
    ]
    records = []
    dc_year = dc["dc_year"].to_numpy()
    dc_group = dc["dc_group"].to_numpy()
    dc_lat = dc["dc_lat"].to_numpy()
    dc_lon = dc["dc_lon"].to_numpy()

    dc_index_by_key = {}
    for scope, _, dc_key_col in region_specs:
        valid = dc[dc_key_col].notna() & dc[dc_key_col].astype(str).ne("")
        dc_index_by_key[scope] = dc.loc[valid].groupby(dc_key_col).indices

    for row in plant_year.itertuples(index=False):
        rec = {"plant_code": row.plant_code, "year": int(row.year)}
        for scope, plant_key_col, _ in region_specs:
            key = getattr(row, plant_key_col)
            if pd.isna(key) or str(key) == "" or not np.isfinite(row.latitude) or not np.isfinite(row.longitude):
                for group in ["all", "larger", "other"]:
                    for window in TIME_WINDOWS:
                        rec[f"dce_{group}_{scope}_{window}"] = np.nan
                        rec[f"exp_{group}_{scope}_{window}"] = np.nan
                continue

            indices = dc_index_by_key[scope].get(key, [])
            if len(indices) == 0:
                weights = np.array([])
                years = np.array([])
                groups = np.array([])
            else:
                indices = np.array(list(indices), dtype=int)
                dist = distance_km(row.latitude, row.longitude, dc_lat[indices], dc_lon[indices])
                weights = 1.0 / (np.maximum(dist, MIN_DISTANCE_KM) / 10.0)
                years = dc_year[indices]
                groups = dc_group[indices]

            for window in TIME_WINDOWS:
                for group in ["all", "larger", "other"]:
                    val, exposed = exposure_value(years, weights, groups, row.year, window, group)
                    rec[f"dce_{group}_{scope}_{window}"] = val
                    rec[f"exp_{group}_{scope}_{window}"] = exposed
        records.append(rec)
    return pd.DataFrame(records)

def add_aggregate_dce(df: pd.DataFrame) -> pd.DataFrame:
    for b in BUFFER_DISTANCES_KM:
        for group in ["all", "larger", "other"]:
            cols = [f"dce_{group}_{b}km_{w}" for w in TIME_WINDOWS]
            exp_cols = [f"exp_{group}_{b}km_{w}" for w in TIME_WINDOWS]
            if all(c in df.columns for c in cols):
                df[f"dceagg_{group}_{b}km"] = df[cols].fillna(0).sum(axis=1)
                df[f"expagg_{group}_{b}km"] = (df[exp_cols].fillna(0).sum(axis=1) > 0).astype(int)

    for scope in ["gid1", "gid2", "ezone"]:
        for group in ["all", "larger", "other"]:
            cols = [f"dce_{group}_{scope}_{w}" for w in TIME_WINDOWS]
            exp_cols = [f"exp_{group}_{scope}_{w}" for w in TIME_WINDOWS]
            if all(c in df.columns for c in cols):
                df[f"dceagg_{group}_{scope}"] = df[cols].fillna(0).sum(axis=1)
                df[f"expagg_{group}_{scope}"] = (df[exp_cols].fillna(0).sum(axis=1) > 0).astype(int)
    return df


def merge_state_controls(df: pd.DataFrame) -> pd.DataFrame:
    if not STATE_CONTROLS.exists():
        df["state_controls_available"] = 0
        return df
    controls = pd.read_csv(STATE_CONTROLS)
    controls["state"] = controls["state"].astype(str).str.strip().str.upper()
    controls["year"] = controls["year"].astype(int)
    df = df.merge(controls, on=["state", "year"], how="left", validate="many_to_one")
    control_vars = [c for c in controls.columns if c not in ["state", "year"]]
    df["state_controls_available"] = df[control_vars].notna().all(axis=1).astype(int)
    return df


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if BASE_PANEL_CSV.exists() and not REBUILD_EIA_BASE_PANEL:
        print(f"Loading existing EIA base panel: {BASE_PANEL_CSV}")
        panel = pd.read_csv(BASE_PANEL_CSV)
    else:
        panel = build_base_panel()

    panel["plant_code"] = panel["plant_code"].map(norm_plant)
    panel = assign_stable_plant_locations(panel)
    dc = load_data_centers()
    plant_year_for_dce = (
        panel[["plant_code", "year", "stable_latitude", "stable_longitude"]]
        .drop_duplicates(["plant_code", "year"])
        .rename(columns={
            "stable_latitude": "latitude",
            "stable_longitude": "longitude",
        })
    )
    print(f"Calculating buffer DCE for {len(plant_year_for_dce):,} plant-years using stable plant coordinates")
    buffer_dce = calculate_buffer_dce(plant_year_for_dce, dc)

    print("Assigning GID and electricity-zone keys")
    plant_year_region, dc_region = add_spatial_region_keys(plant_year_for_dce, dc)
    print("Calculating GID-1, GID-2 and electricity-zone DCE")
    region_dce = calculate_region_dce(plant_year_region, dc_region)

    dce = buffer_dce.merge(region_dce, on=["plant_code", "year"], how="left", validate="one_to_one")
    dce.to_csv(DCE_COVERAGE_CSV, index=False)
    out = panel.merge(dce, on=["plant_code", "year"], how="left", validate="many_to_one")
    out = add_aggregate_dce(out)
    out = merge_state_controls(out)
    out["plant_cf_reg_sample_controls"] = (
        (out["plant_cf_regression_sample"] == 1)
        & (out["state_controls_available"].fillna(0).astype(int) == 1)
    ).astype(int)
    out.to_csv(OUTPUT_CSV, index=False)
    out.to_stata(OUTPUT_DTA, write_index=False, version=118)
    print(f"Saved: {OUTPUT_CSV}")
    print(f"Saved: {OUTPUT_DTA}")
    print("Coverage by year:")
    display(out.groupby("year").agg(
        plants=("plant_code", "nunique"),
        reg_rows=("plant_cf_regression_sample", "sum"),
        mean_cf=("plant_capacity_factor", "mean"),
        exposed_25_2020=("exp_all_25km_20_24", "sum"),
    ).reset_index())


In [ ]:
main()
